# M00 — 環境設定與心智模型

這是整套課程的起點。本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

目標：
1. 確認 Python 版本與套件都裝好。
2. 用供應商無關的 `get_model()` 跑出第一個模型呼叫。
3. 在心裡建立「LangChain = 標準介面 + 可組合組件」的圖。

## 1. 環境檢查

先確認兩件事：Python 是否為 3.10+，以及 `langchain` 是否裝好。
這格用 try/except 包起來，就算還沒裝套件也不會炸掉，而是給你友善提示。

預期輸出（環境正常時）：印出 Python 版本與 langchain 版本，最後一行 `環境檢查通過`。

In [ ]:
import sys

# Check Python version: v1 needs 3.10+.
print(f"Python 版本：{sys.version.split()[0]}")
if sys.version_info < (3, 10):
    print("⚠️  Python 版本過舊，請改用 3.10 以上。")
else:
    print("Python 版本 OK (3.10+)")

# Check that langchain is installed; if not, point the student at requirements.txt.
try:
    import langchain

    print(f"langchain 版本：{langchain.__version__}")
    print("環境檢查通過")
except ImportError:
    print("⚠️  尚未安裝 langchain。請在課程根目錄執行：")
    print("    pip install -r requirements.txt")

## 2. 載入共用 helper 並建立模型

本課程所有 notebook 都透過 `_shared/course_utils.py` 取得模型，
這樣換供應商時程式碼一字不動（細節見 README「供應商無關」段落）。

- `load_env()`：往上找課程根目錄的 `.env`，載入 `COURSE_MODEL` 等變數。
- `get_model()`：底層呼叫 `init_chat_model(...)`，回傳一個對話模型。

⚠️ 若還沒做過 `cp .env.example .env` 並填金鑰，下一格實際呼叫時才會報錯。

In [ ]:
# Add the shared helpers folder to the import path.
# cwd is .../第一冊.../M00_...  ->  parents[1] is the curriculum root that holds _shared/
import sys, pathlib

sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()  # read COURSE_MODEL from the repo-root .env
model = get_model()  # provider-agnostic chat model
print(f"已建立模型物件：{type(model).__name__}")
# Expected output: 已建立模型物件：ChatOpenAI （或你 .env 指定的供應商對應 class）

## 3. 你的第一個模型呼叫

用 `.invoke()` 送一句話給模型，回傳一個 `AIMessage`，文字在 `.content`。
這就是整套課程最核心的一個動作：**輸入訊息 → 模型 → 回覆**。

預期輸出：模型用一句話自我介紹（內容每次略有不同）。

In [ ]:
response = model.invoke("用一句話介紹你自己")
print(response.content)
# Expected output (示意): 我是一個由大型語言模型驅動的 AI 助理，能回答問題並協助各種任務。

## 4. 供應商無關：切換 COURSE_MODEL

重點觀念：上面那行 `get_model()` 完全沒寫死任何供應商。
你**不需要改程式碼**，只要改 `.env` 裡的一行 `COURSE_MODEL`，就能整套切換。

下面示範如何在程式裡臨時指定不同供應商（正式使用請改 `.env`，不要寫死在 notebook）。
把你想試的那行取消註解即可——前提是已安裝對應套件並填好金鑰。

In [ ]:
# All three lines below build a model the *same way*; only the provider:name string differs.
# 正式做法是改 .env 的 COURSE_MODEL，這裡只是示範「介面一致、供應商可換」。

# model = get_model("openai:gpt-4o-mini")        # OpenAI（需 OPENAI_API_KEY）
# model = get_model("anthropic:claude-haiku-4-5")  # Claude（需 ANTHROPIC_API_KEY）
# model = get_model("ollama:llama3.1")           # 本地 Ollama，免金鑰

# 不論上面選哪一個，呼叫方式都一樣：
# print(model.invoke("你好").content)

print("換供應商只需改字串，呼叫方式 (.invoke) 完全相同。")

## 5. 心智圖：LangChain 組件分工

把 LangChain 想成樂高底板，下面這些積木各司其職、接口一致、可以用 `|` 疊起來。
你現在不必懂全部，只要記得「每塊各做一件事，後面模組會逐一拆解」。

```
                       ┌──────────────────────────────┐
                       │          LangChain            │
                       │   標準介面 + 可組合組件        │
                       └──────────────────────────────┘

  prompt  ──►  model  ──►  output parser          ← 「呼叫一次」基本三件套
  (M02)        (M01)        (M02/M03)                 用 LCEL 的 | 串起來 (M03)

    ▲            ▲
    │            │
  tool         retriever                           ← 讓模型有「外部能力 / 外部知識」
  (M04)        (M05)

                 agent                              ← 整合上述全部，自動決策 (M06)
                 └─ 跑在 LangGraph 上（第二冊做有狀態編排）
```

一句話分工：
- **LangChain（第一冊）**：提供組件，把它們串成「呼叫一次」的直線管線。
- **LangGraph（第二冊）**：需要迴圈 / 分支 / 記憶 / 多 Agent 時，做有狀態的編排。

## 🧪 練習 1：改 prompt

把第 3 格的提示詞換成你自己的問題，重新執行，觀察 `.content` 的變化。
例如：請模型用「三點條列」說明什麼是大型語言模型。

In [ ]:
# TODO: 改成你自己的問題
my_question = "用三點條列說明什麼是大型語言模型"
answer = model.invoke(my_question)
print(answer.content)

## 🧪 練習 2：切換模型（供應商無關驗證）

在第 4 格挑一個你已裝好、也填好金鑰的供應商，取消註解、重新跑一次第 3 格。
確認：**呼叫程式碼沒變，只有 COURSE_MODEL / 字串變了**，輸出卻來自不同模型。
這就是「標準介面」帶來的好處——換供應商的成本趨近於零。

## 小結 & 下一步

你完成了：
- 環境檢查（Python 3.10+、langchain 已安裝）。
- 用供應商無關的 `get_model()` 跑出第一個 `.invoke()` 呼叫。
- 建立「LangChain = 標準介面 + 可組合組件」與「LangChain vs LangGraph」的心智模型。

下一個模組 **M01 — 對話模型與訊息**：把鏡頭拉近到 `model` 這塊積木，
認識 `HumanMessage` / `SystemMessage` / `AIMessage` 等訊息種類、
`.invoke()` 與 `.stream()` 的差別，以及多模態輸入。那是後續所有組件的共同基礎。